In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis: Tb2TiO7, HEiDi

This tutorial demonstrates a practical two-stage workflow for single-crystal
diffraction analysis with EasyDiffraction.

In the first stage, we run a fast local refinement to obtain a sensible
point estimate and parameter uncertainties. In the second stage, we use
these refined values to define fit bounds and then sample the posterior
distribution with BUMPS-DREAM.

The example uses constant-wavelength neutron single-crystal diffraction data
for Tb2TiO7 measured on HEiDi at FRM II.

## Import Library

In [ ]:
import easydiffraction as ed

## Step 1: Create a Project Container

In [ ]:
project = ed.Project()

## Step 2: Build the Structural Model

In [ ]:
structure_path = ed.download_data(id=20, destination='data')

In [ ]:
project.structures.add_from_cif_path(structure_path)

In [ ]:
structure = project.structures['tbti']

## Step 3: Define the Diffraction Experiment

In [ ]:
data_path = ed.download_data(id=19, destination='data')

In [ ]:
project.experiments.add_from_data_path(
    name='heidi',
    data_path=data_path,
    sample_form='single crystal',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

In [ ]:
experiment = project.experiments['heidi']

In [ ]:
experiment.linked_crystal.id = 'tbti'
experiment.linked_crystal.scale = 1.0

In [ ]:
experiment.instrument.setup_wavelength = 0.793

In [ ]:
experiment.extinction.mosaicity = 35000
experiment.extinction.radius = 10

## Step 4: Run an Initial Local Refinement

In [ ]:
structure.atom_sites['O1'].fract_x.free = True

structure.atom_sites['Ti'].occupancy.free = False
structure.atom_sites['O1'].occupancy.free = False
structure.atom_sites['O2'].occupancy.free = False

structure.atom_sites['Tb'].adp_iso.free = True
structure.atom_sites['Ti'].adp_iso.free = True
structure.atom_sites['O1'].adp_iso.free = True
structure.atom_sites['O2'].adp_iso.free = True

In [ ]:
experiment.linked_crystal.scale.free = True
experiment.extinction.radius.free = True

In [ ]:
project.analysis.fit.show_minimizer_types()

In [ ]:
project.analysis.fit()

In [ ]:
project.analysis.display.fit_results()

In [ ]:
project.display.plotter.plot_param_correlations()

In [ ]:
project.display.plotter.plot_meas_vs_calc(expt_name='heidi')

## Step 5: Prepare for Bayesian Sampling

In [ ]:
project.analysis.display.free_params()

In [ ]:
for param in project.free_parameters:
    param.set_fit_bounds_from_uncertainty(multiplier=1.5)

In [ ]:
project.analysis.display.free_params()

## Step 6: Configure and Run BUMPS-DREAM

In [ ]:
project.analysis.fit.show_minimizer_types()

In [ ]:
project.analysis.fit.minimizer_type = 'bumps (dream)'

In [ ]:
project.analysis.fit.minimizer.steps = 500

In [ ]:
project.analysis.fit()

In [ ]:
project.analysis.display.fit_results()

In [ ]:
project.display.plotter.plot_param_correlations()

In [ ]:
project.display.plotter.plot_posterior_pairs()

In [ ]:
for param in project.free_parameters:
    project.display.plotter.plot_param_distribution(param)

In [ ]:
project.display.plotter.plot_posterior_predictive(expt_name='heidi')